In [ ]:
# ==============================================================================
# CELL 1: SETUP, IMPORTS & HALO PROPERTIES
# ==============================================================================
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LogNorm, Normalize
from matplotlib.collections import LineCollection
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from IPython.display import HTML, display

import fake_spectra.spectra as spectra
import fake_spectra.unitsystem as unitsystem

# --- 1. MATPLOTLIB PROFESSIONAL STYLE ---
def matplotlib_animation_config():
    """Applies a clean, high-contrast style optimized for animations."""
    font = {'family' : 'serif', 'weight' : 'normal', 'size'   : 14}
    rcnew = {
        "mathtext.fontset" : "cm", "text.usetex": False, 'figure.frameon': True,
        'axes.linewidth': 2., "axes.titlesize" : 20, "axes.labelsize" : 18,
        'xtick.direction': 'in', 'ytick.direction': 'in', 'xtick.top': True, 'ytick.right': True,
        'xtick.labelsize': 16, 'ytick.labelsize': 16,
    }
    return font, rcnew

font, rcnew = matplotlib_animation_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)

# --- 2. TARGET SELECTION & HALO PROPERTIES ---
base_path = '/home/STORAGE/TNG50-4'
snap_num  = 25  
target_halo_id = 0  # 0 is the most massive halo.

def get_halo_info(base_dir, snap_number, halo_id):
    """Fetches Halo position, mass, R200c, and BoxSize."""
    group_dir = os.path.join(base_dir, f"groups_{snap_number:03d}")
    file0 = os.path.join(group_dir, f"fof_subhalo_tab_{snap_number:03d}.0.hdf5")
    
    with h5py.File(file0, 'r') as f:
        num_files   = f['Header'].attrs['NumFiles']
        total_halos = f['Header'].attrs['Ngroups_Total']
        box_size    = f['Header'].attrs['BoxSize']
        
    if halo_id >= total_halos:
        raise ValueError(f"Invalid ID {halo_id}. Max is {total_halos - 1}.")
        
    current_id_offset = 0
    for i in range(num_files):
        filepath = os.path.join(group_dir, f"fof_subhalo_tab_{snap_number:03d}.{i}.hdf5")
        with h5py.File(filepath, 'r') as f:
            if 'Group' not in f or 'GroupMass' not in f['Group']: continue
            n_halos = len(f['Group/GroupMass'])
            if halo_id < current_id_offset + n_halos:
                local_index = halo_id - current_id_offset
                return (f['Group/GroupPos'][local_index], 
                        f['Group/Group_M_Crit200'][local_index], 
                        f['Group/Group_R_Crit200'][local_index], 
                        box_size, num_files)
            current_id_offset += n_halos

halo_pos, halo_mass, halo_r200, box_size, num_files_cat = get_halo_info(base_path, snap_num, target_halo_id)

# Extract Cosmology
snap_file_0 = os.path.join(base_path, f'snapdir_{snap_num:03d}', f'snap_{snap_num:03d}.0.hdf5')
with h5py.File(snap_file_0, 'r') as f:
    header = f['Header'].attrs
    a_scale = header.get('Time', 1.0)
    z_sim = header.get('Redshift', (1.0/a_scale) - 1.0)
    num_snap_files = header.get('NumFilesPerSnapshot', header.get('NumFiles', 1))

print(f"Target Halo ID: {target_halo_id}")
print(f"Mass (M200c)  : {halo_mass:.2e} x 10^10 M_sun/h")
print(f"Halo R200     : {halo_r200:.2f} ckpc/h")
print(f"Box Size      : {box_size:.2f} ckpc/h")

In [ ]:
# ==============================================================================
# CELL 2: EXTRACT 2D GAS MAP (FULL BOX - SHARED BY BOTH ANIMATIONS)
# ==============================================================================
print("\nReading snapshot to generate 2D Context Map for the entire box...")
slice_half_width = 1000.0  # ckpc/h thickness in Z around the halo
z_halo = halo_pos[2]

x_gas, y_gas, mass_gas = [], [], []

for i in range(num_snap_files):
    snap_file = os.path.join(base_path, f'snapdir_{snap_num:03d}', f'snap_{snap_num:03d}.{i}.hdf5')
    if not os.path.exists(snap_file): continue
    try:
        with h5py.File(snap_file, 'r') as f:
            if 'PartType0' not in f: continue
            coords = f['PartType0']['Coordinates'][:]
            masses = f['PartType0']['Masses'][:]
            
            # Apply Periodic Boundary Conditions for the Z-slice
            dz = np.abs(coords[:, 2] - z_halo)
            dz = np.where(dz > box_size / 2.0, box_size - dz, dz)
            mask = dz <= slice_half_width
            
            if np.sum(mask) > 0:
                x_gas.append(coords[:, 0][mask])
                y_gas.append(coords[:, 1][mask])
                mass_gas.append(masses[mask])
    except Exception:
        pass

# Convert to cMpc/h for plotting
x_gas_c = np.concatenate(x_gas) / 1000.0  
y_gas_c = np.concatenate(y_gas) / 1000.0
mass_gas_c = np.concatenate(mass_gas)
box_cmpc = box_size / 1000.0
halo_x_cmpc = halo_pos[0] / 1000.0
halo_y_cmpc = halo_pos[1] / 1000.0

print(f"Gas map ready. Extracted {len(x_gas_c):,} particles spanning the whole box.")

In [ ]:
# ==============================================================================
# CELL 3: ANIMATION 1 — SKEWER CROSSING THE TARGET HALO
# ==============================================================================
print("Extracting Spectra (LOS through Target Halo)...")

# Setup Skewer: Spans X-axis. Y and Z are fixed exactly at the halo's center.
cofm_halo = np.array([[0.0, halo_pos[1], halo_pos[2]]])
axis_x = np.array([1], dtype=np.int32)
tng_units = unitsystem.UnitSystem(UnitLength_in_cm=3.085678e21, UnitMass_in_g=1.989e43, UnitVelocity_in_cm_per_s=1e5)

save_file_halo = f"anim_skewer_halo{target_halo_id}.hdf5"

# Run Spectra
ss_halo = spectra.Spectra(
    snap_num, base_path, cofm=cofm_halo, axis=axis_x, res=2.0, 
    savedir=".", savefile=save_file_halo, reload_file=True, kernel=None,
    units=tng_units, sf_neutral=True, filter_mode="None"
)

flux_halo = np.exp(-ss_halo.get_tau("H", 1, 1215)[0, :])
nbins = ss_halo.nbins
chi_axis = np.linspace(0, box_cmpc, nbins)

if os.path.exists(save_file_halo): os.remove(save_file_halo)

print("Building Animation...")
# --- Setup Figure ---
fig1 = plt.figure(figsize=(12, 10))
gs1 = fig1.add_gridspec(2, 1, height_ratios=[1.5, 1], hspace=0.1)
ax_map1 = fig1.add_subplot(gs1[0])
ax_spec1 = fig1.add_subplot(gs1[1], sharex=ax_map1)

# --- Aesthetic Map ---
cmap_bg = plt.get_cmap('inferno')
ax_map1.set_facecolor(cmap_bg(0)) # Fixes the black dots from empty bins
h_map, xedges, yedges, img1 = ax_map1.hist2d(
    x_gas_c, y_gas_c, bins=600, range=[[0, box_cmpc], [0, box_cmpc]], 
    weights=mass_gas_c, cmap=cmap_bg, norm=LogNorm(vmin=1e-5, vmax=1e-1)
)

# Halo Marker
ax_map1.scatter(halo_x_cmpc, halo_y_cmpc, marker='*', facecolor='none', s=350, 
                edgecolor='cyan', lw=1.5, zorder=5, label=f'Target Halo {target_halo_id}')

# Colormap mapping for the moving line
cmap_line = plt.get_cmap('turbo')
norm_line = Normalize(vmin=0, vmax=box_cmpc)

# Line Collections
lc_map1 = LineCollection([], cmap=cmap_line, norm=norm_line, lw=3.5, zorder=10)
ax_map1.add_collection(lc_map1)
tip_map1, = ax_map1.plot([], [], 'o', color='white', markersize=10, markeredgecolor='black', markeredgewidth=1.5, zorder=11)

ax_map1.set_ylabel(r'$Y \ [\mathrm{cMpc}/h]$')
ax_map1.set_title(f'LOS Intersecting Halo {target_halo_id} (z={z_sim:.2f})', pad=15)
ax_map1.legend(loc='upper right', facecolor='white', framealpha=0.8, fontsize=12)

# Perfect Colorbar Alignment
divider1 = make_axes_locatable(ax_map1)
cax1 = divider1.append_axes("right", size="3%", pad=0.1)
cbar1 = fig1.colorbar(img1, cax=cax1)
cbar1.set_label(r'Gas Mass Density', color='black')

# --- Bottom Spectrum Panel ---
ax_spec1.set_facecolor('#ffffff')
ax_spec1.set_xlim(0, box_cmpc)
ax_spec1.set_ylim(-0.05, 1.05)
ax_spec1.spines['top'].set_visible(False)
ax_spec1.spines['right'].set_visible(False)

ax_spec1.set_xlabel(r'Distance along LOS $X \ [\mathrm{cMpc}/h]$')
ax_spec1.set_ylabel('Transmitted Flux')
ax_spec1.grid(True, alpha=0.3, linestyle='-')

# Reference lines
ax_spec1.axhline(1.0, color='gray', linestyle=':', lw=1.5, alpha=0.5)
ax_spec1.axhline(0.0, color='gray', linestyle='-', lw=1.5, alpha=0.3)

lc_spec1 = LineCollection([], cmap=cmap_line, norm=norm_line, lw=2.5)
ax_spec1.add_collection(lc_spec1)
tip_spec1, = ax_spec1.plot([], [], 'o', color='white', markersize=8, markeredgecolor='black', markeredgewidth=1.5)

# Drop-down guide line connecting the tip to the axis
guide_line1, = ax_spec1.plot([], [], color='gray', linestyle=':', lw=1.5, alpha=0.7)

ax_spec1.axvline(halo_x_cmpc, color='gray', linestyle='--', lw=1.5, alpha=0.5)
ax_spec1.text(halo_x_cmpc + 0.3, 0.2, 'Halo Center', color='gray', rotation=90, fontsize=12)

# --- Animation Logic ---
N_FRAMES = 150

def init1():
    lc_map1.set_segments([])
    lc_spec1.set_segments([])
    tip_map1.set_data([], [])
    tip_spec1.set_data([], [])
    guide_line1.set_data([], [])
    return lc_map1, lc_spec1, tip_map1, tip_spec1, guide_line1

def animate1(frame):
    idx = int((frame + 1) * nbins / N_FRAMES)
    if idx == 0: idx = 1
    
    x_cur = chi_axis[:idx]
    y_cur_map = np.full_like(x_cur, halo_y_cmpc)
    y_cur_spec = flux_halo[:idx]
    
    # Map line
    pts_map = np.array([x_cur, y_cur_map]).T.reshape(-1, 1, 2)
    lc_map1.set_segments(np.concatenate([pts_map[:-1], pts_map[1:]], axis=1))
    lc_map1.set_array(x_cur[:-1]) 
    
    # Spectrum line
    pts_spec = np.array([x_cur, y_cur_spec]).T.reshape(-1, 1, 2)
    lc_spec1.set_segments(np.concatenate([pts_spec[:-1], pts_spec[1:]], axis=1))
    lc_spec1.set_array(x_cur[:-1])
    
    # Markers & Guide
    current_color = cmap_line(norm_line(x_cur[-1]))
    tip_map1.set_data([x_cur[-1]], [y_cur_map[-1]])
    tip_map1.set_markerfacecolor(current_color)
    
    tip_spec1.set_data([x_cur[-1]], [y_cur_spec[-1]])
    tip_spec1.set_markerfacecolor(current_color)
    
    guide_line1.set_data([x_cur[-1], x_cur[-1]], [-0.05, y_cur_spec[-1]])
    
    return lc_map1, lc_spec1, tip_map1, tip_spec1, guide_line1

anim1 = FuncAnimation(fig1, animate1, frames=N_FRAMES, init_func=init1, blit=True, interval=50)
plt.close(fig1)

display(HTML(anim1.to_jshtml()))
output_gif1 = f"LOS_Halo{target_halo_id}.gif"
anim1.save(output_gif1, writer=PillowWriter(fps=20))
print(f"Saved: {output_gif1}")

In [ ]:
# ==============================================================================
# CELL 1: SETUP, IMPORTS & HALO PROPERTIES
# ==============================================================================
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LogNorm, Normalize
from matplotlib.collections import LineCollection
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from IPython.display import HTML, display

import fake_spectra.spectra as spectra
import fake_spectra.unitsystem as unitsystem

# --- 1. MATPLOTLIB PROFESSIONAL STYLE ---
def matplotlib_animation_config():
    """Applies a clean, high-contrast style optimized for animations."""
    font = {'family' : 'serif', 'weight' : 'normal', 'size'   : 14}
    rcnew = {
        "mathtext.fontset" : "cm", "text.usetex": False, 'figure.frameon': True,
        'axes.linewidth': 2., "axes.titlesize" : 20, "axes.labelsize" : 18,
        'xtick.direction': 'in', 'ytick.direction': 'in', 'xtick.top': True, 'ytick.right': True,
        'xtick.labelsize': 16, 'ytick.labelsize': 16,
    }
    return font, rcnew

font, rcnew = matplotlib_animation_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)

# --- 2. TARGET SELECTION & HALO PROPERTIES ---
base_path = '/home/STORAGE/TNG50-4'
snap_num  = 25  
target_halo_id = 0  # 0 is the most massive halo.

def get_halo_info(base_dir, snap_number, halo_id):
    """Fetches Halo position, mass, R200c, and BoxSize."""
    group_dir = os.path.join(base_dir, f"groups_{snap_number:03d}")
    file0 = os.path.join(group_dir, f"fof_subhalo_tab_{snap_number:03d}.0.hdf5")
    
    with h5py.File(file0, 'r') as f:
        num_files   = f['Header'].attrs['NumFiles']
        total_halos = f['Header'].attrs['Ngroups_Total']
        box_size    = f['Header'].attrs['BoxSize']
        
    if halo_id >= total_halos:
        raise ValueError(f"Invalid ID {halo_id}. Max is {total_halos - 1}.")
        
    current_id_offset = 0
    for i in range(num_files):
        filepath = os.path.join(group_dir, f"fof_subhalo_tab_{snap_number:03d}.{i}.hdf5")
        with h5py.File(filepath, 'r') as f:
            if 'Group' not in f or 'GroupMass' not in f['Group']: continue
            n_halos = len(f['Group/GroupMass'])
            if halo_id < current_id_offset + n_halos:
                local_index = halo_id - current_id_offset
                return (f['Group/GroupPos'][local_index], 
                        f['Group/Group_M_Crit200'][local_index], 
                        f['Group/Group_R_Crit200'][local_index], 
                        box_size, num_files)
            current_id_offset += n_halos

halo_pos, halo_mass, halo_r200, box_size, num_files_cat = get_halo_info(base_path, snap_num, target_halo_id)

# Extract Cosmology
snap_file_0 = os.path.join(base_path, f'snapdir_{snap_num:03d}', f'snap_{snap_num:03d}.0.hdf5')
with h5py.File(snap_file_0, 'r') as f:
    header = f['Header'].attrs
    a_scale = header.get('Time', 1.0)
    z_sim = header.get('Redshift', (1.0/a_scale) - 1.0)
    num_snap_files = header.get('NumFilesPerSnapshot', header.get('NumFiles', 1))

print(f"Target Halo ID: {target_halo_id}")
print(f"Mass (M200c)  : {halo_mass:.2e} x 10^10 M_sun/h")
print(f"Halo R200     : {halo_r200:.2f} ckpc/h")
print(f"Box Size      : {box_size:.2f} ckpc/h")
# ==============================================================================
# CELL 2: EXTRACT 2D GAS MAP (FULL BOX - SHARED BY BOTH ANIMATIONS)
# ==============================================================================
print("\nReading snapshot to generate 2D Context Map for the entire box...")
slice_half_width = 1000.0  # ckpc/h thickness in Z around the halo
z_halo = halo_pos[2]

x_gas, y_gas, mass_gas = [], [], []

for i in range(num_snap_files):
    snap_file = os.path.join(base_path, f'snapdir_{snap_num:03d}', f'snap_{snap_num:03d}.{i}.hdf5')
    if not os.path.exists(snap_file): continue
    try:
        with h5py.File(snap_file, 'r') as f:
            if 'PartType0' not in f: continue
            coords = f['PartType0']['Coordinates'][:]
            masses = f['PartType0']['Masses'][:]
            
            # Apply Periodic Boundary Conditions for the Z-slice
            dz = np.abs(coords[:, 2] - z_halo)
            dz = np.where(dz > box_size / 2.0, box_size - dz, dz)
            mask = dz <= slice_half_width
            
            if np.sum(mask) > 0:
                x_gas.append(coords[:, 0][mask])
                y_gas.append(coords[:, 1][mask])
                mass_gas.append(masses[mask])
    except Exception:
        pass

# Convert to cMpc/h for plotting
x_gas_c = np.concatenate(x_gas) / 1000.0  
y_gas_c = np.concatenate(y_gas) / 1000.0
mass_gas_c = np.concatenate(mass_gas)
box_cmpc = box_size / 1000.0
halo_x_cmpc = halo_pos[0] / 1000.0
halo_y_cmpc = halo_pos[1] / 1000.0

print(f"Gas map ready. Extracted {len(x_gas_c):,} particles spanning the whole box.")
# ==============================================================================
# CELL 3: ANIMATION 1 — SKEWER CROSSING THE TARGET HALO
# ==============================================================================
print("Extracting Spectra (LOS through Target Halo)...")

# Setup Skewer: Spans X-axis. Y and Z are fixed exactly at the halo's center.
cofm_halo = np.array([[0.0, halo_pos[1], halo_pos[2]]])
axis_x = np.array([1], dtype=np.int32)
tng_units = unitsystem.UnitSystem(UnitLength_in_cm=3.085678e21, UnitMass_in_g=1.989e43, UnitVelocity_in_cm_per_s=1e5)

save_file_halo = f"anim_skewer_halo{target_halo_id}.hdf5"

# Run Spectra
ss_halo = spectra.Spectra(
    snap_num, base_path, cofm=cofm_halo, axis=axis_x, res=2.0, 
    savedir=".", savefile=save_file_halo, reload_file=True, kernel=None,
    units=tng_units, sf_neutral=True, filter_mode="None"
)

flux_halo = np.exp(-ss_halo.get_tau("H", 1, 1215)[0, :])
nbins = ss_halo.nbins
chi_axis = np.linspace(0, box_cmpc, nbins)

if os.path.exists(save_file_halo): os.remove(save_file_halo)

print("Building Animation...")
# --- Setup Figure ---
fig1 = plt.figure(figsize=(12, 10))
gs1 = fig1.add_gridspec(2, 1, height_ratios=[1.5, 1], hspace=0.1)
ax_map1 = fig1.add_subplot(gs1[0])
ax_spec1 = fig1.add_subplot(gs1[1], sharex=ax_map1)

# --- Aesthetic Map ---
cmap_bg = plt.get_cmap('inferno')
ax_map1.set_facecolor(cmap_bg(0)) # Fixes the black dots from empty bins
h_map, xedges, yedges, img1 = ax_map1.hist2d(
    x_gas_c, y_gas_c, bins=600, range=[[0, box_cmpc], [0, box_cmpc]], 
    weights=mass_gas_c, cmap=cmap_bg, norm=LogNorm(vmin=1e-5, vmax=1e-1)
)

# Halo Marker
ax_map1.scatter(halo_x_cmpc, halo_y_cmpc, marker='*', facecolor='none', s=350, 
                edgecolor='cyan', lw=1.5, zorder=5, label=f'Target Halo {target_halo_id}')

# Colormap mapping for the moving line
cmap_line = plt.get_cmap('turbo')
norm_line = Normalize(vmin=0, vmax=box_cmpc)

# Line Collections
lc_map1 = LineCollection([], cmap=cmap_line, norm=norm_line, lw=3.5, zorder=10)
ax_map1.add_collection(lc_map1)
tip_map1, = ax_map1.plot([], [], 'o', color='white', markersize=10, markeredgecolor='black', markeredgewidth=1.5, zorder=11)

ax_map1.set_ylabel(r'$Y \ [\mathrm{cMpc}/h]$')
ax_map1.set_title(f'LOS Intersecting Halo {target_halo_id} (z={z_sim:.2f})', pad=15)
ax_map1.legend(loc='upper right', facecolor='white', framealpha=0.8, fontsize=12)

# Perfect Colorbar Alignment
divider1 = make_axes_locatable(ax_map1)
cax1 = divider1.append_axes("right", size="3%", pad=0.1)
cbar1 = fig1.colorbar(img1, cax=cax1)
cbar1.set_label(r'Gas Mass Density', color='black')

# --- Bottom Spectrum Panel ---
ax_spec1.set_facecolor('#ffffff')
ax_spec1.set_xlim(0, box_cmpc)
ax_spec1.set_ylim(-0.05, 1.05)
ax_spec1.spines['top'].set_visible(False)
ax_spec1.spines['right'].set_visible(False)

ax_spec1.set_xlabel(r'Distance along LOS $X \ [\mathrm{cMpc}/h]$')
ax_spec1.set_ylabel('Transmitted Flux')
ax_spec1.grid(True, alpha=0.3, linestyle='-')

# Reference lines
ax_spec1.axhline(1.0, color='gray', linestyle=':', lw=1.5, alpha=0.5)
ax_spec1.axhline(0.0, color='gray', linestyle='-', lw=1.5, alpha=0.3)

lc_spec1 = LineCollection([], cmap=cmap_line, norm=norm_line, lw=2.5)
ax_spec1.add_collection(lc_spec1)
tip_spec1, = ax_spec1.plot([], [], 'o', color='white', markersize=8, markeredgecolor='black', markeredgewidth=1.5)

# Drop-down guide line connecting the tip to the axis
guide_line1, = ax_spec1.plot([], [], color='gray', linestyle=':', lw=1.5, alpha=0.7)

ax_spec1.axvline(halo_x_cmpc, color='gray', linestyle='--', lw=1.5, alpha=0.5)
ax_spec1.text(halo_x_cmpc + 0.3, 0.2, 'Halo Center', color='gray', rotation=90, fontsize=12)

# --- Animation Logic ---
N_FRAMES = 150

def init1():
    lc_map1.set_segments([])
    lc_spec1.set_segments([])
    tip_map1.set_data([], [])
    tip_spec1.set_data([], [])
    guide_line1.set_data([], [])
    return lc_map1, lc_spec1, tip_map1, tip_spec1, guide_line1

def animate1(frame):
    idx = int((frame + 1) * nbins / N_FRAMES)
    if idx == 0: idx = 1
    
    x_cur = chi_axis[:idx]
    y_cur_map = np.full_like(x_cur, halo_y_cmpc)
    y_cur_spec = flux_halo[:idx]
    
    # Map line
    pts_map = np.array([x_cur, y_cur_map]).T.reshape(-1, 1, 2)
    lc_map1.set_segments(np.concatenate([pts_map[:-1], pts_map[1:]], axis=1))
    lc_map1.set_array(x_cur[:-1]) 
    
    # Spectrum line
    pts_spec = np.array([x_cur, y_cur_spec]).T.reshape(-1, 1, 2)
    lc_spec1.set_segments(np.concatenate([pts_spec[:-1], pts_spec[1:]], axis=1))
    lc_spec1.set_array(x_cur[:-1])
    
    # Markers & Guide
    current_color = cmap_line(norm_line(x_cur[-1]))
    tip_map1.set_data([x_cur[-1]], [y_cur_map[-1]])
    tip_map1.set_markerfacecolor(current_color)
    
    tip_spec1.set_data([x_cur[-1]], [y_cur_spec[-1]])
    tip_spec1.set_markerfacecolor(current_color)
    
    guide_line1.set_data([x_cur[-1], x_cur[-1]], [-0.05, y_cur_spec[-1]])
    
    return lc_map1, lc_spec1, tip_map1, tip_spec1, guide_line1

anim1 = FuncAnimation(fig1, animate1, frames=N_FRAMES, init_func=init1, blit=True, interval=50)
plt.close(fig1)

display(HTML(anim1.to_jshtml()))
output_gif1 = f"LOS_Halo{target_halo_id}.gif"
anim1.save(output_gif1, writer=PillowWriter(fps=20))
print(f"Saved: {output_gif1}")
# ==============================================================================
# FULL ANIMATION: TRIPTYCH DECOMPOSITION (MAP, TOTAL+HCD, FOREST)
# ==============================================================================
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LogNorm, Normalize
from matplotlib.collections import LineCollection
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from IPython.display import HTML, display

import fake_spectra.spectra as spectra
import fake_spectra.unitsystem as unitsystem

# --- 0. CONFIGURATION & MATPLOTLIB STYLE ---
def matplotlib_animation_config():
    font = {'family' : 'serif', 'weight' : 'normal', 'size'   : 14}
    rcnew = {
        "mathtext.fontset" : "cm", "text.usetex": False, 'figure.frameon': True,
        'axes.linewidth': 2., "axes.titlesize" : 18, "axes.labelsize" : 16,
        'xtick.direction': 'in', 'ytick.direction': 'in', 'xtick.top': True, 'ytick.right': True,
        'xtick.labelsize': 14, 'ytick.labelsize': 14,
    }
    return font, rcnew

font, rcnew = matplotlib_animation_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)

# --- 1. TARGET SELECTION & HALO PROPERTIES ---
base_path = '/home/STORAGE/TNG50-4'
snap_num  = 25  
target_halo_id = 0  

def get_halo_info(base_dir, snap_number, halo_id):
    group_dir = os.path.join(base_dir, f"groups_{snap_number:03d}")
    file0 = os.path.join(group_dir, f"fof_subhalo_tab_{snap_number:03d}.0.hdf5")
    with h5py.File(file0, 'r') as f:
        num_files   = f['Header'].attrs['NumFiles']
        total_halos = f['Header'].attrs['Ngroups_Total']
        box_size    = f['Header'].attrs['BoxSize']
        
    current_id_offset = 0
    for i in range(num_files):
        filepath = os.path.join(group_dir, f"fof_subhalo_tab_{snap_number:03d}.{i}.hdf5")
        with h5py.File(filepath, 'r') as f:
            if 'Group' not in f or 'GroupMass' not in f['Group']: continue
            n_halos = len(f['Group/GroupMass'])
            if halo_id < current_id_offset + n_halos:
                local_index = halo_id - current_id_offset
                return (f['Group/GroupPos'][local_index], 
                        f['Group/Group_M_Crit200'][local_index], 
                        f['Group/Group_R_Crit200'][local_index], 
                        box_size, num_files)
            current_id_offset += n_halos

halo_pos, halo_mass, halo_r200, box_size, num_files_cat = get_halo_info(base_path, snap_num, target_halo_id)

snap_file_0 = os.path.join(base_path, f'snapdir_{snap_num:03d}', f'snap_{snap_num:03d}.0.hdf5')
with h5py.File(snap_file_0, 'r') as f:
    a_scale = f['Header'].attrs.get('Time', 1.0)
    z_sim = f['Header'].attrs.get('Redshift', (1.0/a_scale) - 1.0)
    num_snap_files = f['Header'].attrs.get('NumFilesPerSnapshot', f['Header'].attrs.get('NumFiles', 1))

print(f"Target Halo ID: {target_halo_id} | Box Size: {box_size/1000.0:.2f} cMpc/h")

# --- 2. EXTRACT 2D GAS MAP (FULL BOX) ---
print("Reading snapshot to generate 2D Context Map...")
slice_half_width = 1000.0  
z_halo = halo_pos[2]
x_gas, y_gas, mass_gas = [], [], []

for i in range(num_snap_files):
    snap_file = os.path.join(base_path, f'snapdir_{snap_num:03d}', f'snap_{snap_num:03d}.{i}.hdf5')
    if not os.path.exists(snap_file): continue
    try:
        with h5py.File(snap_file, 'r') as f:
            if 'PartType0' not in f: continue
            coords = f['PartType0']['Coordinates'][:]
            masses = f['PartType0']['Masses'][:]
            dz = np.abs(coords[:, 2] - z_halo)
            dz = np.where(dz > box_size / 2.0, box_size - dz, dz)
            mask = dz <= slice_half_width
            if np.sum(mask) > 0:
                x_gas.append(coords[:, 0][mask])
                y_gas.append(coords[:, 1][mask])
                mass_gas.append(masses[mask])
    except Exception:
        pass

x_gas_c = np.concatenate(x_gas) / 1000.0  
y_gas_c = np.concatenate(y_gas) / 1000.0
mass_gas_c = np.concatenate(mass_gas)
box_cmpc = box_size / 1000.0
halo_x_cmpc = halo_pos[0] / 1000.0
halo_y_cmpc = halo_pos[1] / 1000.0

# --- 3. EXTRACT ALL 3 MODES OF THE SKEWER ---
print("Running fake_spectra for the sightline crossing the Halo in 3 modes...")

cofm_halo = np.array([[0.0, halo_pos[1], halo_pos[2]]])
axis_x = np.array([1], dtype=np.int32)
tng_units = unitsystem.UnitSystem(UnitLength_in_cm=3.085678e21, UnitMass_in_g=1.989e43, UnitVelocity_in_cm_per_s=1e5)

# Mode 1: TOTAL (None)
ss_total = spectra.Spectra(
    snap_num, base_path, cofm=cofm_halo, axis=axis_x, res=2.0, 
    savedir=".", savefile="dummy_total.hdf5", reload_file=True, kernel=None,
    units=tng_units, sf_neutral=True, filter_mode="None"
)
flux_total = np.exp(-ss_total.get_tau("H", 1, 1215)[0, :])
nbins = ss_total.nbins
chi_axis = np.linspace(0, box_cmpc, nbins)

# Mode 2: HCD ONLY (DLAs)
ss_hcd = spectra.Spectra(
    snap_num, base_path, cofm=cofm_halo, axis=axis_x, res=2.0, 
    savedir=".", savefile="dummy_hcd.hdf5", reload_file=True, kernel=None,
    units=tng_units, sf_neutral=True, filter_mode="DLAs", dla_phys_thresh="auto"
)
flux_hcd = np.exp(-ss_hcd.get_tau("H", 1, 1215)[0, :])

# Mode 3: FOREST ONLY (No_DLAs)
ss_forest = spectra.Spectra(
    snap_num, base_path, cofm=cofm_halo, axis=axis_x, res=2.0, 
    savedir=".", savefile="dummy_forest.hdf5", reload_file=True, kernel=None,
    units=tng_units, sf_neutral=True, filter_mode="No_DLAs", dla_phys_thresh="auto"
)
flux_forest = np.exp(-ss_forest.get_tau("H", 1, 1215)[0, :])

for f in ["dummy_total.hdf5", "dummy_hcd.hdf5", "dummy_forest.hdf5"]:
    if os.path.exists(f): os.remove(f)

# --- 4. BUILD THE TRIPTYCH ANIMATION ---
print("Building Triptych Animation...")
fig = plt.figure(figsize=(14, 12))
gs = fig.add_gridspec(3, 1, height_ratios=[1.8, 1, 1], hspace=0.1)
ax_map   = fig.add_subplot(gs[0])
ax_spec1 = fig.add_subplot(gs[1], sharex=ax_map)
ax_spec2 = fig.add_subplot(gs[2], sharex=ax_map)

# --- MAP PANEL ---
cmap_bg = plt.get_cmap('inferno')
ax_map.set_facecolor(cmap_bg(0)) 
h_map, xedges, yedges, img = ax_map.hist2d(
    x_gas_c, y_gas_c, bins=600, range=[[0, box_cmpc], [0, box_cmpc]], 
    weights=mass_gas_c, cmap=cmap_bg, norm=LogNorm(vmin=1e-5, vmax=1e-1)
)

ax_map.scatter(halo_x_cmpc, halo_y_cmpc, marker='*', facecolor='none', s=450, 
                edgecolor='cyan', lw=1.5, zorder=5, label=f'Target Halo {target_halo_id}')

cmap_line = plt.get_cmap('turbo')
norm_line = Normalize(vmin=0, vmax=box_cmpc)
lc_map = LineCollection([], cmap=cmap_line, norm=norm_line, lw=3.5, zorder=10)
ax_map.add_collection(lc_map)
tip_map, = ax_map.plot([], [], 'o', color='white', markersize=10, markeredgecolor='black', markeredgewidth=1.5, zorder=11)

ax_map.set_ylabel(r'$Y \ [\mathrm{cMpc}/h]$')
ax_map.set_title(f'Spectral Decomposition Sweep (z={z_sim:.2f})', pad=15)
ax_map.legend(loc='upper right', facecolor='white', framealpha=0.8)

divider = make_axes_locatable(ax_map)
cax = divider.append_axes("right", size="2.5%", pad=0.1)
cbar = fig.colorbar(img, cax=cax)
cbar.set_label(r'Gas Mass Density')

# --- MIDDLE PANEL: TOTAL & HCD ---
ax_spec1.set_facecolor('#ffffff')
ax_spec1.set_xlim(0, box_cmpc)
ax_spec1.set_ylim(-0.05, 1.05)
ax_spec1.spines['top'].set_visible(False)
ax_spec1.spines['right'].set_visible(False)
ax_spec1.spines['bottom'].set_visible(False) # Hide axis to connect to bottom panel
ax_spec1.tick_params(labelbottom=False, bottom=False)

ax_spec1.set_ylabel('Transmitted Flux')
ax_spec1.grid(True, alpha=0.3, linestyle='-')
ax_spec1.axhline(1.0, color='gray', linestyle=':', lw=1.5, alpha=0.5)

lc_total = LineCollection([], colors='gray', lw=2.5, alpha=0.6, label='Total Spectrum')
lc_hcd   = LineCollection([], colors='crimson', lw=2.5, label='HCD Component')
ax_spec1.add_collection(lc_total)
ax_spec1.add_collection(lc_hcd)

tip_total, = ax_spec1.plot([], [], 'o', color='gray', markersize=6)
tip_hcd,   = ax_spec1.plot([], [], 'o', color='crimson', markersize=8, markeredgecolor='black')
guide_1,   = ax_spec1.plot([], [], color='gray', linestyle=':', lw=1.5, alpha=0.7)

ax_spec1.axvline(halo_x_cmpc, color='gray', linestyle='--', lw=1.5, alpha=0.5)
ax_spec1.legend(loc='lower right', framealpha=0.9)

# --- BOTTOM PANEL: FOREST ONLY ---
ax_spec2.set_facecolor('#ffffff')
ax_spec2.set_xlim(0, box_cmpc)
ax_spec2.set_ylim(-0.05, 1.05)
ax_spec2.spines['top'].set_visible(False)
ax_spec2.spines['right'].set_visible(False)

ax_spec2.set_xlabel(r'Distance along LOS $X \ [\mathrm{cMpc}/h]$')
ax_spec2.set_ylabel('Transmitted Flux')
ax_spec2.grid(True, alpha=0.3, linestyle='-')
ax_spec2.axhline(1.0, color='gray', linestyle=':', lw=1.5, alpha=0.5)

lc_forest = LineCollection([], colors='forestgreen', lw=2.5, label='Forest Component')
ax_spec2.add_collection(lc_forest)
tip_forest, = ax_spec2.plot([], [], 'o', color='forestgreen', markersize=8, markeredgecolor='black')
guide_2,    = ax_spec2.plot([], [], color='gray', linestyle=':', lw=1.5, alpha=0.7)

ax_spec2.axvline(halo_x_cmpc, color='gray', linestyle='--', lw=1.5, alpha=0.5)
ax_spec2.legend(loc='lower right', framealpha=0.9)

# --- ANIMATION LOGIC ---
N_FRAMES = 150

def init():
    lc_map.set_segments([])
    lc_total.set_segments([])
    lc_hcd.set_segments([])
    lc_forest.set_segments([])
    
    tip_map.set_data([], [])
    tip_total.set_data([], [])
    tip_hcd.set_data([], [])
    tip_forest.set_data([], [])

In [ ]:
# ==============================================================================
# FULL SCRIPT: TNG50-4 LYMAN-ALPHA ANIMATIONS (SINGLE & MULTI-PANEL)
# ==============================================================================
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LogNorm, Normalize
from matplotlib.collections import LineCollection
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from IPython.display import HTML, display

import fake_spectra.spectra as spectra
import fake_spectra.unitsystem as unitsystem

# --- 1. MATPLOTLIB PROFESSIONAL STYLE ---
def matplotlib_animation_config():
    """Applies a clean, high-contrast style optimized for animations."""
    font = {'family' : 'serif', 'weight' : 'normal', 'size'   : 14}
    rcnew = {
        "mathtext.fontset" : "cm", "text.usetex": False, 'figure.frameon': True,
        'axes.linewidth': 2., "axes.titlesize" : 18, "axes.labelsize" : 16,
        'xtick.direction': 'in', 'ytick.direction': 'in', 'xtick.top': True, 'ytick.right': True,
        'xtick.labelsize': 14, 'ytick.labelsize': 14,
    }
    return font, rcnew

font, rcnew = matplotlib_animation_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)

# --- 2. TARGET SELECTION & HALO PROPERTIES ---
base_path = '/home/STORAGE/TNG50-4'
snap_num  = 25  
target_halo_id = 0  # 0 is the most massive halo.

def get_halo_info(base_dir, snap_number, halo_id):
    """Fetches Halo position, mass, R200c, and BoxSize."""
    group_dir = os.path.join(base_dir, f"groups_{snap_number:03d}")
    file0 = os.path.join(group_dir, f"fof_subhalo_tab_{snap_number:03d}.0.hdf5")
    
    with h5py.File(file0, 'r') as f:
        num_files   = f['Header'].attrs['NumFiles']
        total_halos = f['Header'].attrs['Ngroups_Total']
        box_size    = f['Header'].attrs['BoxSize']
        
    if halo_id >= total_halos:
        raise ValueError(f"Invalid ID {halo_id}. Max is {total_halos - 1}.")
        
    current_id_offset = 0
    for i in range(num_files):
        filepath = os.path.join(group_dir, f"fof_subhalo_tab_{snap_number:03d}.{i}.hdf5")
        with h5py.File(filepath, 'r') as f:
            if 'Group' not in f or 'GroupMass' not in f['Group']: continue
            n_halos = len(f['Group/GroupMass'])
            if halo_id < current_id_offset + n_halos:
                local_index = halo_id - current_id_offset
                return (f['Group/GroupPos'][local_index], 
                        f['Group/Group_M_Crit200'][local_index], 
                        f['Group/Group_R_Crit200'][local_index], 
                        box_size, num_files)
            current_id_offset += n_halos

halo_pos, halo_mass, halo_r200, box_size, num_files_cat = get_halo_info(base_path, snap_num, target_halo_id)

# Extract Cosmology
snap_file_0 = os.path.join(base_path, f'snapdir_{snap_num:03d}', f'snap_{snap_num:03d}.0.hdf5')
with h5py.File(snap_file_0, 'r') as f:
    header = f['Header'].attrs
    a_scale = header.get('Time', 1.0)
    z_sim = header.get('Redshift', (1.0/a_scale) - 1.0)
    num_snap_files = header.get('NumFilesPerSnapshot', header.get('NumFiles', 1))

print(f"Target Halo ID: {target_halo_id}")
print(f"Mass (M200c)  : {halo_mass:.2e} x 10^10 M_sun/h")
print(f"Halo R200     : {halo_r200:.2f} ckpc/h")
print(f"Box Size      : {box_size:.2f} ckpc/h")

# --- 3. EXTRACT 2D GAS MAP (FULL BOX - SHARED BY BOTH ANIMATIONS) ---
print("\nReading snapshot to generate 2D Context Map for the entire box...")
slice_half_width = 1000.0  # ckpc/h thickness in Z around the halo
z_halo = halo_pos[2]

x_gas, y_gas, mass_gas = [], [], []

for i in range(num_snap_files):
    snap_file = os.path.join(base_path, f'snapdir_{snap_num:03d}', f'snap_{snap_num:03d}.{i}.hdf5')
    if not os.path.exists(snap_file): continue
    try:
        with h5py.File(snap_file, 'r') as f:
            if 'PartType0' not in f: continue
            coords = f['PartType0']['Coordinates'][:]
            masses = f['PartType0']['Masses'][:]
            
            # Apply Periodic Boundary Conditions for the Z-slice
            dz = np.abs(coords[:, 2] - z_halo)
            dz = np.where(dz > box_size / 2.0, box_size - dz, dz)
            mask = dz <= slice_half_width
            
            if np.sum(mask) > 0:
                x_gas.append(coords[:, 0][mask])
                y_gas.append(coords[:, 1][mask])
                mass_gas.append(masses[mask])
    except Exception:
        pass

# Convert to cMpc/h for plotting
x_gas_c = np.concatenate(x_gas) / 1000.0  
y_gas_c = np.concatenate(y_gas) / 1000.0
mass_gas_c = np.concatenate(mass_gas)
box_cmpc = box_size / 1000.0
halo_x_cmpc = halo_pos[0] / 1000.0
halo_y_cmpc = halo_pos[1] / 1000.0

print(f"Gas map ready. Extracted {len(x_gas_c):,} particles spanning the whole box.")


# ==============================================================================
# ANIMATION 1 — SINGLE PANEL (TOTAL SPECTRUM CROSSING THE TARGET HALO)
# ==============================================================================
print("\n--- Generating Animation 1 (Single Panel Spectrum) ---")

# Setup Skewer: Spans X-axis. Y and Z are fixed exactly at the halo's center.
cofm_halo = np.array([[0.0, halo_pos[1], halo_pos[2]]])
axis_x = np.array([1], dtype=np.int32)
tng_units = unitsystem.UnitSystem(UnitLength_in_cm=3.085678e21, UnitMass_in_g=1.989e43, UnitVelocity_in_cm_per_s=1e5)

save_file_halo = f"anim_skewer_halo{target_halo_id}.hdf5"

# Run Spectra in "None" mode (full physical spectrum)
ss_halo = spectra.Spectra(
    snap_num, base_path, cofm=cofm_halo, axis=axis_x, res=2.0, 
    savedir=".", savefile=save_file_halo, reload_file=True, kernel=None,
    units=tng_units, sf_neutral=True, filter_mode="None"
)

flux_halo = np.exp(-ss_halo.get_tau("H", 1, 1215)[0, :])
nbins = ss_halo.nbins
chi_axis = np.linspace(0, box_cmpc, nbins)

if os.path.exists(save_file_halo): os.remove(save_file_halo)

# --- Setup Figure ---
fig1 = plt.figure(figsize=(12, 10))
gs1 = fig1.add_gridspec(2, 1, height_ratios=[1.5, 1], hspace=0.1)
ax_map1 = fig1.add_subplot(gs1[0])
ax_spec1 = fig1.add_subplot(gs1[1], sharex=ax_map1)

# --- Aesthetic Map ---
cmap_bg = plt.get_cmap('inferno')
ax_map1.set_facecolor(cmap_bg(0)) # Fixes the black dots from empty bins
h_map, xedges, yedges, img1 = ax_map1.hist2d(
    x_gas_c, y_gas_c, bins=600, range=[[0, box_cmpc], [0, box_cmpc]], 
    weights=mass_gas_c, cmap=cmap_bg, norm=LogNorm(vmin=1e-5, vmax=1e-1)
)

# Halo Marker
ax_map1.scatter(halo_x_cmpc, halo_y_cmpc, marker='*', facecolor='none', s=350, 
                edgecolor='cyan', lw=1.5, zorder=5, label=f'Target Halo {target_halo_id}')

# Colormap mapping for the moving line
cmap_line = plt.get_cmap('turbo')
norm_line = Normalize(vmin=0, vmax=box_cmpc)

# Line Collections
lc_map1 = LineCollection([], cmap=cmap_line, norm=norm_line, lw=3.5, zorder=10)
ax_map1.add_collection(lc_map1)
tip_map1, = ax_map1.plot([], [], 'o', color='white', markersize=10, markeredgecolor='black', markeredgewidth=1.5, zorder=11)

ax_map1.set_ylabel(r'$Y \ [\mathrm{cMpc}/h]$')
ax_map1.set_title(f'LOS Intersecting Halo {target_halo_id} (z={z_sim:.2f})', pad=15)
ax_map1.legend(loc='upper right', facecolor='white', framealpha=0.8, fontsize=12)

# Perfect Colorbar Alignment
divider1 = make_axes_locatable(ax_map1)
cax1 = divider1.append_axes("right", size="3%", pad=0.1)
cbar1 = fig1.colorbar(img1, cax=cax1)
cbar1.set_label(r'Gas Mass Density', color='black')

# --- Bottom Spectrum Panel ---
ax_spec1.set_facecolor('#ffffff')
ax_spec1.set_xlim(0, box_cmpc)
ax_spec1.set_ylim(-0.05, 1.05)
ax_spec1.spines['top'].set_visible(False)
ax_spec1.spines['right'].set_visible(False)

ax_spec1.set_xlabel(r'Distance along LOS $X \ [\mathrm{cMpc}/h]$')
ax_spec1.set_ylabel('Transmitted Flux')
ax_spec1.grid(True, alpha=0.3, linestyle='-')

# Reference lines
ax_spec1.axhline(1.0, color='gray', linestyle=':', lw=1.5, alpha=0.5)
ax_spec1.axhline(0.0, color='gray', linestyle='-', lw=1.5, alpha=0.3)

lc_spec1 = LineCollection([], cmap=cmap_line, norm=norm_line, lw=2.5)
ax_spec1.add_collection(lc_spec1)
tip_spec1, = ax_spec1.plot([], [], 'o', color='white', markersize=8, markeredgecolor='black', markeredgewidth=1.5)

# Drop-down guide line connecting the tip to the axis
guide_line1, = ax_spec1.plot([], [], color='gray', linestyle=':', lw=1.5, alpha=0.7)

ax_spec1.axvline(halo_x_cmpc, color='gray', linestyle='--', lw=1.5, alpha=0.5)
ax_spec1.text(halo_x_cmpc + 0.3, 0.2, 'Halo Center', color='gray', rotation=90, fontsize=12)

# --- Animation Logic ---
N_FRAMES = 150

def init1():
    lc_map1.set_segments([])
    lc_spec1.set_segments([])
    tip_map1.set_data([], [])
    tip_spec1.set_data([], [])
    guide_line1.set_data([], [])
    return lc_map1, lc_spec1, tip_map1, tip_spec1, guide_line1

def animate1(frame):
    idx = int((frame + 1) * nbins / N_FRAMES)
    if idx == 0: idx = 1
    
    x_cur = chi_axis[:idx]
    y_cur_map = np.full_like(x_cur, halo_y_cmpc)
    y_cur_spec = flux_halo[:idx]
    
    # Map line
    pts_map = np.array([x_cur, y_cur_map]).T.reshape(-1, 1, 2)
    lc_map1.set_segments(np.concatenate([pts_map[:-1], pts_map[1:]], axis=1))
    lc_map1.set_array(x_cur[:-1]) 
    
    # Spectrum line
    pts_spec = np.array([x_cur, y_cur_spec]).T.reshape(-1, 1, 2)
    lc_spec1.set_segments(np.concatenate([pts_spec[:-1], pts_spec[1:]], axis=1))
    lc_spec1.set_array(x_cur[:-1])
    
    # Markers & Guide
    current_color = cmap_line(norm_line(x_cur[-1]))
    tip_map1.set_data([x_cur[-1]], [y_cur_map[-1]])
    tip_map1.set_markerfacecolor(current_color)
    
    tip_spec1.set_data([x_cur[-1]], [y_cur_spec[-1]])
    tip_spec1.set_markerfacecolor(current_color)
    
    guide_line1.set_data([x_cur[-1], x_cur[-1]], [-0.05, y_cur_spec[-1]])
    
    return lc_map1, lc_spec1, tip_map1, tip_spec1, guide_line1

anim1 = FuncAnimation(fig1, animate1, frames=N_FRAMES, init_func=init1, blit=True, interval=50)
plt.close(fig1)

display(HTML(anim1.to_jshtml()))
output_gif1 = f"LOS_Halo{target_halo_id}_Single.gif"
anim1.save(output_gif1, writer=PillowWriter(fps=20))
print(f"Saved: {output_gif1}")


# ==============================================================================
# ANIMATION 2 — MULTI-PANEL (TOTAL, HCD ONLY, FOREST ONLY)
# ==============================================================================
print("\n--- Generating Animation 2 (Triptych Decomposition) ---")

# Run Spectra for HCD Only
ss_hcd = spectra.Spectra(
    snap_num, base_path, cofm=cofm_halo, axis=axis_x, res=2.0, 
    savedir=".", savefile="dummy_hcd.hdf5", reload_file=True, kernel=None,
    units=tng_units, sf_neutral=True, filter_mode="DLAs", dla_phys_thresh="auto"
)
flux_hcd = np.exp(-ss_hcd.get_tau("H", 1, 1215)[0, :])

# Run Spectra for Forest Only
ss_forest = spectra.Spectra(
    snap_num, base_path, cofm=cofm_halo, axis=axis_x, res=2.0, 
    savedir=".", savefile="dummy_forest.hdf5", reload_file=True, kernel=None,
    units=tng_units, sf_neutral=True, filter_mode="No_DLAs", dla_phys_thresh="auto"
)
flux_forest = np.exp(-ss_forest.get_tau("H", 1, 1215)[0, :])

for f in ["dummy_hcd.hdf5", "dummy_forest.hdf5"]:
    if os.path.exists(f): os.remove(f)

# --- Setup Figure (4 panels: 1 Map + 3 Spectra) ---
fig2 = plt.figure(figsize=(16, 9))
gs2 = fig2.add_gridspec(4, 1, height_ratios=[1.8, 1, 1, 1], hspace=0.1)
ax_map2 = fig2.add_subplot(gs2[0])
ax_tot  = fig2.add_subplot(gs2[1], sharex=ax_map2)
ax_hcd  = fig2.add_subplot(gs2[2], sharex=ax_map2)
ax_for  = fig2.add_subplot(gs2[3], sharex=ax_map2)

# --- MAP PANEL ---
ax_map2.set_facecolor(cmap_bg(0)) 
h_map2, xedges, yedges, img2 = ax_map2.hist2d(
    x_gas_c, y_gas_c, bins=600, range=[[0, box_cmpc], [0, box_cmpc]], 
    weights=mass_gas_c, cmap=cmap_bg, norm=LogNorm(vmin=1e-5, vmax=1e-1)
)

ax_map2.scatter(halo_x_cmpc, halo_y_cmpc, marker='*', facecolor='none', s=450, 
                edgecolor='cyan', lw=1.5, zorder=5, label=f'Target Halo {target_halo_id}')

lc_map2 = LineCollection([], cmap=cmap_line, norm=norm_line, lw=3.5, zorder=10)
ax_map2.add_collection(lc_map2)
tip_map2, = ax_map2.plot([], [], 'o', color='white', markersize=10, markeredgecolor='black', markeredgewidth=1.5, zorder=11)

ax_map2.set_ylabel(r'$Y \ [\mathrm{cMpc}/h]$')
ax_map2.set_title(f'Spectral Decomposition Sweep (z={z_sim:.2f})', pad=15)
ax_map2.legend(loc='upper right', facecolor='white', framealpha=0.8)

divider2 = make_axes_locatable(ax_map2)
cax2 = divider2.append_axes("right", size="2.5%", pad=0.1)
cbar2 = fig2.colorbar(img2, cax=cax2)
cbar2.set_label(r'Gas Mass Density')

# --- CONFIG FOR ALL SPECTRA PANELS ---
for ax in [ax_tot, ax_hcd, ax_for]:
    ax.set_facecolor('#ffffff')
    ax.set_xlim(0, box_cmpc)
    ax.set_ylim(-0.05, 1.05)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylabel('Flux')
    ax.grid(True, alpha=0.3, linestyle='-')
    ax.axhline(1.0, color='gray', linestyle=':', lw=1.5, alpha=0.5)
    ax.axvline(halo_x_cmpc, color='gray', linestyle='--', lw=1.5, alpha=0.5)

# Hide x-labels for upper spectra panels to keep them tightly stacked
ax_tot.spines['bottom'].set_visible(False)
ax_tot.tick_params(labelbottom=False, bottom=False)
ax_hcd.spines['bottom'].set_visible(False)
ax_hcd.tick_params(labelbottom=False, bottom=False)
ax_for.set_xlabel(r'Distance along LOS $X \ [\mathrm{cMpc}/h]$')

# Panel 1: Total
lc_tot = LineCollection([], cmap=cmap_line, norm=norm_line, lw=2.5, label='Total Spectrum')
ax_tot.add_collection(lc_tot)
tip_tot, = ax_tot.plot([], [], 'o', color='white', markersize=6, markeredgecolor='black')
guide_tot, = ax_tot.plot([], [], color='gray', linestyle=':', lw=1.5, alpha=0.7)
ax_tot.legend(loc='lower right', framealpha=0.9)

# Panel 2: HCD
lc_hcd = LineCollection([], cmap=cmap_line, norm=norm_line, lw=2.5, label='HCD Only')
ax_hcd.add_collection(lc_hcd)
tip_hcd, = ax_hcd.plot([], [], 'o', color='white', markersize=6, markeredgecolor='black')
guide_hcd, = ax_hcd.plot([], [], color='gray', linestyle=':', lw=1.5, alpha=0.7)
ax_hcd.legend(loc='lower right', framealpha=0.9)

# Panel 3: Forest
lc_for = LineCollection([], cmap=cmap_line, norm=norm_line, lw=2.5, label='Forest Only')
ax_for.add_collection(lc_for)
tip_for, = ax_for.plot([], [], 'o', color='white', markersize=6, markeredgecolor='black')
guide_for, = ax_for.plot([], [], color='gray', linestyle=':', lw=1.5, alpha=0.7)
ax_for.legend(loc='lower right', framealpha=0.9)

# --- ANIMATION LOGIC ---
def init2():
    lc_map2.set_segments([])
    lc_tot.set_segments([])
    lc_hcd.set_segments([])
    lc_for.set_segments([])
    
    tip_map2.set_data([], [])
    tip_tot.set_data([], [])
    tip_hcd.set_data([], [])
    tip_for.set_data([], [])
    
    guide_tot.set_data([], [])
    guide_hcd.set_data([], [])
    guide_for.set_data([], [])
    
    return (lc_map2, lc_tot, lc_hcd, lc_for, 
            tip_map2, tip_tot, tip_hcd, tip_for, 
            guide_tot, guide_hcd, guide_for)

def animate2(frame):
    idx = int((frame + 1) * nbins / N_FRAMES)
    if idx == 0: idx = 1
    
    x_cur = chi_axis[:idx]
    y_cur_map = np.full_like(x_cur, halo_y_cmpc)
    
    current_color = cmap_line(norm_line(x_cur[-1]))
    
    # Update Map
    pts_map = np.array([x_cur, y_cur_map]).T.reshape(-1, 1, 2)
    lc_map2.set_segments(np.concatenate([pts_map[:-1], pts_map[1:]], axis=1))
    lc_map2.set_array(x_cur[:-1]) 
    tip_map2.set_data([x_cur[-1]], [y_cur_map[-1]])
    tip_map2.set_markerfacecolor(current_color)
    
    # Helper to update spectra
    def update_spectrum(y_array, lc_obj, tip_obj, guide_obj):
        y_cur = y_array[:idx]
        pts = np.array([x_cur, y_cur]).T.reshape(-1, 1, 2)
        lc_obj.set_segments(np.concatenate([pts[:-1], pts[1:]], axis=1))
        lc_obj.set_array(x_cur[:-1])
        tip_obj.set_data([x_cur[-1]], [y_cur[-1]])
        tip_obj.set_markerfacecolor(current_color)
        guide_obj.set_data([x_cur[-1], x_cur[-1]], [-0.05, 1.05])
        
    update_spectrum(flux_halo, lc_tot, tip_tot, guide_tot)
    update_spectrum(flux_hcd, lc_hcd, tip_hcd, guide_hcd)
    update_spectrum(flux_forest, lc_for, tip_for, guide_for)
    
    return (lc_map2, lc_tot, lc_hcd, lc_for, 
            tip_map2, tip_tot, tip_hcd, tip_for, 
            guide_tot, guide_hcd, guide_for)

anim2 = FuncAnimation(fig2, animate2, frames=N_FRAMES, init_func=init2, blit=True, interval=50)
plt.close(fig2)

display(HTML(anim2.to_jshtml()))
output_gif2 = f"LOS_Decomposition_Halo{target_halo_id}.gif"
print(f"Saving animation to {output_gif2} ...")
anim2.save(output_gif2, writer=PillowWriter(fps=20))
print("Done! Both animations are ready.")